# Phase 1A: Raw Dataset Inspection

This notebook inspects raw CSV files in `datasets/raw/` and writes a summary report to `reports/raw_data_report.txt`.

**Important:** This notebook does not modify any data.

In [2]:
from pathlib import Path
import sys
import pandas as pd


def resolve_project_root() -> Path:
    cwd = Path.cwd()
    if (cwd / "soil_ai_system").exists():
        return cwd / "soil_ai_system"
    if cwd.name == "notebooks":
        return cwd.parent
    return cwd


project_root = resolve_project_root()
sys.path.append(str(project_root))

from config import RAW_DATA_PATH, REPORT_PATH, RAW_DATA_REPORT_FILENAME

raw_dir = (project_root / RAW_DATA_PATH).resolve()
report_path = (project_root / REPORT_PATH / RAW_DATA_REPORT_FILENAME).resolve()

print(f"Raw data directory: {raw_dir}")
print(f"Report path: {report_path}")

Raw data directory: C:\Users\vinay\PROJECTS\soil-intelligence-system\soil_ai_system\datasets\raw
Report path: C:\Users\vinay\PROJECTS\soil-intelligence-system\soil_ai_system\reports\raw_data_report.txt


In [ ]:
datasets = {}

csv_files = sorted(raw_dir.glob("*.csv"))
if not csv_files:
    print("No CSV files found in raw data directory.")

for path in csv_files:
    datasets[path.name] = pd.read_csv(path)
    print(f"Loaded {path.name}: {datasets[path.name].shape}")

Loaded crop_recommendation: (2200, 8)
Loaded soil_fertility: (880, 13)
Missing datasets:
- C:\Users\vinay\PROJECTS\soil-intelligence-system\soil_ai_system\datasets\raw\optional_india_soil_data.csv


In [4]:
def inspect_dataset(name: str, df: pd.DataFrame) -> dict:
    summary = {
        'shape': df.shape,
        'columns': list(df.columns),
        'dtypes': df.dtypes.astype(str).to_dict(),
        'null_counts': df.isnull().sum().to_dict(),
        'duplicate_rows': int(df.duplicated().sum()),
        'sample_rows': df.head(5).to_dict(orient='records')
    }
    return summary

summaries = {name: inspect_dataset(name, df) for name, df in datasets.items()}
summaries

{'crop_recommendation': {'shape': (2200, 8),
  'columns': ['N',
   'P',
   'K',
   'temperature',
   'humidity',
   'ph',
   'rainfall',
   'label'],
  'dtypes': {'N': 'int64',
   'P': 'int64',
   'K': 'int64',
   'temperature': 'float64',
   'humidity': 'float64',
   'ph': 'float64',
   'rainfall': 'float64',
   'label': 'str'},
  'null_counts': {'N': 0,
   'P': 0,
   'K': 0,
   'temperature': 0,
   'humidity': 0,
   'ph': 0,
   'rainfall': 0,
   'label': 0},
  'duplicate_rows': 0,
  'sample_rows': [{'N': 90,
    'P': 42,
    'K': 43,
    'temperature': 20.87974371,
    'humidity': 82.00274423,
    'ph': 6.502985292000001,
    'rainfall': 202.9355362,
    'label': 'rice'},
   {'N': 85,
    'P': 58,
    'K': 41,
    'temperature': 21.77046169,
    'humidity': 80.31964408,
    'ph': 7.038096361,
    'rainfall': 226.6555374,
    'label': 'rice'},
   {'N': 60,
    'P': 55,
    'K': 44,
    'temperature': 23.00445915,
    'humidity': 82.3207629,
    'ph': 7.840207144,
    'rainfall': 263.9

In [5]:
def normalize_column(col: str) -> str:
    return ''.join(ch for ch in col.lower() if ch.isalnum())

schema = {name: set(df.columns) for name, df in datasets.items()}
union_cols = set().union(*schema.values()) if schema else set()

missing_by_dataset = {name: sorted(list(union_cols - cols)) for name, cols in schema.items()}

normalized_map = {}
for name, cols in schema.items():
    normalized_map[name] = {normalize_column(col): col for col in cols}

potential_mismatches = []
if len(normalized_map) > 1:
    all_norm = set().union(*[set(m.keys()) for m in normalized_map.values()])
    for norm in all_norm:
        originals = set()
        for m in normalized_map.values():
            if norm in m:
                originals.add(m[norm])
        if len(originals) > 1:
            potential_mismatches.append((norm, sorted(originals)))

missing_by_dataset, potential_mismatches

({'crop_recommendation': ['B',
   'Cu',
   'EC',
   'Fe',
   'Mn',
   'OC',
   'Output',
   'S',
   'Zn',
   'pH'],
  'soil_fertility': ['humidity', 'label', 'ph', 'rainfall', 'temperature']},
 [('ph', ['pH', 'ph'])])

In [6]:
report_lines = []
report_lines.append("Raw Data Inspection Report")
report_lines.append("==========================")
report_lines.append("")

for name, summary in summaries.items():
    report_lines.append(f"Dataset: {name}")
    report_lines.append(f"Shape: {summary['shape']}")
    report_lines.append("Columns: " + ", ".join(summary["columns"]))
    report_lines.append("Dtypes:")
    for col, dtype in summary["dtypes"].items():
        report_lines.append(f"  - {col}: {dtype}")
    report_lines.append("Null Counts:")
    for col, count in summary["null_counts"].items():
        report_lines.append(f"  - {col}: {count}")
    report_lines.append(f"Duplicate Rows: {summary['duplicate_rows']}")
    report_lines.append("Sample Rows:")
    for row in summary["sample_rows"]:
        report_lines.append(f"  - {row}")
    report_lines.append("")

report_lines.append("Schema Differences:")
for name, missing_cols in missing_by_dataset.items():
    report_lines.append(f"- {name} missing {len(missing_cols)} columns: {missing_cols}")

report_lines.append("")
report_lines.append("Potential Column Name Mismatches:")
if potential_mismatches:
    for norm, originals in potential_mismatches:
        report_lines.append(f"- {norm}: {originals}")
else:
    report_lines.append("- None detected")

report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join(report_lines), encoding="utf-8")

print(f"Report saved to {report_path}")

Report saved to C:\Users\vinay\PROJECTS\soil-intelligence-system\soil_ai_system\reports\raw_data_report.txt
